# Simple RAG with LangChain and LangSmith

Minimal retrieval augmented generation pipeline with LangSmith tracing enabled. Run the Colab setup cell or the VS Code setup cell, not both.

In [ ]:
# Setup for Google Colab
!pip install -q langchain langchain-openai langchain-community

import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass("LangSmith API key: ")
os.environ["LANGCHAIN_PROJECT"] = "simple-rag"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
OpenAI API key: ··········
LangSmith API key: ··········


In [ ]:
# # Setup for VS Code (local environment)
# # pip install langchain langchain-openai langchain-community python-dotenv

# import os
# # from dotenv import load_dotenv

# load_dotenv()
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ.setdefault("LANGCHAIN_PROJECT", "simple-rag")
# assert os.environ.get("OPENAI_API_KEY") and os.environ.get("LANGCHAIN_API_KEY")

In [ ]:
pip install pypdf langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.4 MB/s eta 0:00:00


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

# -------------------------------
# Load PDF
# -------------------------------
pdf_path = "sample.pdf"      # Replace with your PDF path

loader = PyPDFLoader(pdf_path)
docs = loader.load()

print(f"📄 Pages Loaded       : {len(docs)}")

# Optional: Total characters in PDF
total_chars = sum(len(doc.page_content) for doc in docs)
print(f"📝 Total Characters   : {total_chars:,}")

# -------------------------------
# Split Documents
# -------------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

print(f"✂️ Chunks Created     : {len(chunks)}")

# -------------------------------
# Create Embedding Model
# -------------------------------
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# Check embedding dimension
sample_embedding = embeddings.embed_query(chunks[0].page_content)

print(f"🧠 Embedding Dimension: {len(sample_embedding)}")
print(f"🧮 Embeddings Created : {len(chunks)}")

# -------------------------------
# Create Vector Store
# -------------------------------
vectorstore = InMemoryVectorStore.from_documents(
    chunks,
    embedding=embeddings
)

print("✅ Vector Store Created Successfully!")

# -------------------------------
# Create Retriever
# -------------------------------
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print("🔍 Retriever Ready!")

/tmp/ipykernel_1224/2716968100.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


📄 Pages Loaded       : 15
📝 Total Characters   : 51,391
✂️ Chunks Created     : 132
🧠 Embedding Dimension: 1536
🧮 Embeddings Created : 132
✅ Vector Store Created Successfully!
🔍 Retriever Ready!


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template(
    """Answer the question using only the context below.

Context:
{context}

Question: {question}
"""
)


def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)


# Print retrieved chunks
def print_and_format_docs(retrieved_docs):
    print("\n" + "=" * 60)
    print(f"📚 Retrieved Chunks: {len(retrieved_docs)}")
    print("=" * 60)

    for i, doc in enumerate(retrieved_docs, start=1):
        print(f"\n🔹 Chunk {i}")
        print("-" * 40)
        print(doc.page_content)

    print("=" * 60 + "\n")

    return format_docs(retrieved_docs)


llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

rag_chain = (
    {
        "context": retriever | print_and_format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
answer = rag_chain.invoke("What is This Document All About used for?")
print(answer)


📚 Retrieved Chunks: 2

🔹 Chunk 1
----------------------------------------
**Chapter 2: The Anatomy of Attention — How Transformers See the World** *Decoding the Core
Mechanism That Changed Everything* Here, we dissect the transformer’s revolutionary engine:
**self-attention**. Through intuitive visuals and step-by-step explanations, we reveal how attention
allows models to dynamically weigh the importance of every word in a sentence—like a reader instantly
connecting a pronoun to its subject across pages. We break down the mechanics: query-key-value

🔹 Chunk 2
----------------------------------------
(Whisper), code generation (Codex), and even protein folding (AlphaFold 2). We explore how the same
architecture adapts: images are “tokenized” into patches, audio into spectrogram segments, and
proteins into amino acid sequences. The narrative emphasizes *transfer learning*—how models
pre-trained on vast text corpora (like GPT-3) can be fine-tuned for specialized tasks with minimal data.